## Visualize Fun Results

Notebook based on 
* `/nfs/turbo/umms-minjilab/sionkim/miajet/notebooks/Bcell_vs_DP.ipynb`



In [1]:
import pandas as pd
import numpy as np
import os

import pybedtools

def precision_recall_f1(tp, fp, fn):
    if tp == 0 and fp == 0 and fn == 0:
        return 1.0, 1.0, 1.0
        
    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

def find_intersections(df_pred_in, df_true_in, leniency):
    """
    The df_pred must have the followign columns ['chrom', 'root', 'unique_id']

    The df_true is assumed to be the loading file given to the 3DPolyS-LE simulation,
    which has columns ['name', 'position', 'length', 'factor']. The only necessary column here is
    the 'position'
    """
    if df_pred_in.empty:
        # No predictions provided, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=0, fn=len(df_true_in) if df_true_in is not None else 0)
        result_dict = {
            "tp": 0,
            "fp": 0,
            "fn": len(df_true_in) if df_true_in is not None else 0,
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return pd.DataFrame(), result_dict

    if df_true_in is None:
        # No true data provided, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=len(df_pred_in), fn=0)
        result_dict = {
            "tp": 0,
            "fp": len(df_pred_in),
            "fn": 0,
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return pd.DataFrame(), result_dict

    df_pred = df_pred_in.copy()
    df_true = df_true_in.copy()

    df_pred["root"] = df_pred["root"].astype(int)

    # Need to map from genomic coordinates to bin coordiantes, which appears to 
    # differ by resolution and factor 2
    df_true["position"] /= 2 
    df_true["position"] = df_true["position"].astype(int)

    # Make BED dataframes for pybedtools intersect
    df_pred_bed = df_pred.assign(
        start=lambda d: np.clip(d["root"] - int(leniency), 0, None),
        end=lambda d: d["root"] + int(leniency) + 1 # half-open BED interval
    )[["chrom", "start", "end", "unique_id", "root"]]


    chrom = df_pred["chrom"].iloc[0]
    df_true_bed = df_true.assign(
        chrom=chrom,
        start=lambda d: np.clip(d["position"] - int(leniency), 0, None),
        end=lambda d: d["position"] + int(leniency) + 1,
    )[["chrom", "start", "end", "name", "position"]]

    pred_bed = pybedtools.BedTool.from_dataframe(df_pred_bed)
    true_bed = pybedtools.BedTool.from_dataframe(df_true_bed)

    hits = pred_bed.intersect(true_bed, wa=True, wb=True)

    intersections = hits.to_dataframe(names=[
        "chrom", "pred_start", "pred_end", "unique_id", "pred_root",
        "true_chrom", "true_start", "true_end", "true_name", "true_position"
    ])

    if intersections.empty:
        # No matches found, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=len(df_pred), fn=len(df_true))
        result_dict = {
            "tp": 0,
            "fp": len(df_pred),
            "fn": len(df_true),
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return intersections, result_dict

    # --- Deduplicate: one-to-one matching by closest distance ---
    intersections["distance"] = (intersections["pred_root"] - intersections["true_position"]).abs()

    # For each true jet, keep only the closest predicted match
    intersections = intersections.sort_values("distance")
    intersections = intersections.drop_duplicates(subset="true_name", keep="first")

    # Also ensure each prediction only maps to one true jet (the closest)
    intersections = intersections.drop_duplicates(subset="unique_id", keep="first")

    # Compute scores
    matched_pred_ids = set(intersections["unique_id"].unique()) # TP
    all_pred_ids = set(df_pred["unique_id"].unique())
    matched_true_names = set(intersections["true_name"].unique()) 
    all_true_names = set(df_true["name"].unique())

    tp = len(matched_pred_ids)
    fp = len(all_pred_ids - matched_pred_ids)
    fn = len(all_true_names - matched_true_names)

    p, r, f = precision_recall_f1(tp=tp, fp=fp, fn=fn)

    result_dict = {
        "tp": tp,
        "fp" : fp,
        "fn" : fn,
        "precision": p,
        "recall": r,
        "f1": f,
    }

    return intersections, result_dict
    

## Computing performance pooled across all simulations 

In [2]:
# Generate performance score for each resolution
resolution = 25000

window_sizes = [
    int(5e6),
    int(5e6),
    int(5e6),
    int(5e6),
    int(5e6),
    int(5e6),
    int(5e6),
    int(5e6),
    int(5e6),
]

hic_files=[
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1b_s-23/out_fast_bounded_jets/r2.14/test1b_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/out_fast_unbounded_jets/r2.14/test1a_s-23-fast_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/out_fast_unbounded_jets/r2.14/chromsim1-fast-unbounded_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/out_fast_bounded_jets/r2.14/chromsim1-fast-bounded_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/out_slow_unbounded_jets/r2.14/test1a_s-23-slow_hic_003.hic", # Should be much smaller window size and highest resolution possible
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1c_s-23/out_fast_bounded_jets/r2.14/test1c_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1cii_s-23/out_fast_bounded_jets/r2.14/test1cii_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1e_s-23/out_tads/r2.14/test1e_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1eii_s-23/out_tads/r2.14/test1eii_s-23_hic_003.hic",
]

signal_noise_ratio = 1.25
bedpe_files = [
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1b_s-23_hic_003_{resolution}_{window_sizes[0]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1a_s-23-fast_hic_003_{resolution}_{window_sizes[1]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/chromsim1-fast-unbounded_hic_003_{resolution}_{window_sizes[2]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/chromsim1-fast-bounded_hic_003_{resolution}_{window_sizes[3]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1a_s-23-slow_hic_003_{resolution}_{window_sizes[4]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1c_s-23_hic_003_{resolution}_{window_sizes[5]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1cii_s-23_hic_003_{resolution}_{window_sizes[6]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1e_s-23_hic_003_{resolution}_{window_sizes[7]}/fountains_{signal_noise_ratio}.bedpe",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1eii_s-23_hic_003_{resolution}_{window_sizes[8]}/fountains_{signal_noise_ratio}.bedpe",
]

tab_files = [
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1b_s-23_hic_003_{resolution}_{window_sizes[0]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1a_s-23-fast_hic_003_{resolution}_{window_sizes[1]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/chromsim1-fast-unbounded_hic_003_{resolution}_{window_sizes[2]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/chromsim1-fast-bounded_hic_003_{resolution}_{window_sizes[3]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1a_s-23-slow_hic_003_{resolution}_{window_sizes[4]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1c_s-23_hic_003_{resolution}_{window_sizes[5]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1cii_s-23_hic_003_{resolution}_{window_sizes[6]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1e_s-23_hic_003_{resolution}_{window_sizes[7]}/fountains_{signal_noise_ratio}.tab",
    f"/nfs/turbo/umms-minjilab/sionkim/Fun_modified/Fountain/test1eii_s-23_hic_003_{resolution}_{window_sizes[8]}/fountains_{signal_noise_ratio}.tab",
]

jet_loc_files = [
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1b_s-23/loading.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/narrow_loading_50.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/JET-fast-unbounded_loading_1_100.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/JET-fast-bounded_loading_1_100.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/diffuse_loading_10000.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1c_s-23/loading.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1cii_s-23/loading.csv",
    None, # no true jets (TADs)
    None, # no true jets (TADs)
]


In [3]:
leniency = 25e3 * 2 # ± Uncertainty for matching predicted fountain to true jet

In [4]:
result_dict_pool = {
    "tp": 0,
    "fp": 0,
    "fn": 0,
    "n_true": 0,
    "n_reported": 0,
}

# FUN
for i in range(len(hic_files)):

    f_hic_file = hic_files[i]
    f_pred_fun_bedpe = bedpe_files[i]
    f_pred_fun_tab = tab_files[i]

    # NEW: load true
    f_true_jet_loc = jet_loc_files[i]
    if f_true_jet_loc is not None:
        df_true = pd.read_csv(f_true_jet_loc)
    else:
        df_true = None

    save_dir = os.path.dirname(f_pred_fun_bedpe)

    chrom = "chrS"

    # Load it in
    fun_bedpe = pd.read_csv(f_pred_fun_bedpe, sep="\t")
    fun_table = pd.read_csv(f_pred_fun_tab, sep="\t")

    # midpoint
    fun_bedpe["extrusion_x"] = (fun_bedpe["y1"] + fun_bedpe["y2"]) / 2
    fun_bedpe["extrusion_y"] = (fun_bedpe["x1"] + fun_bedpe["x2"]) / 2

    # guarantees that the extrusion point is in the upper diagonal
    # doesn't matter except for convention but should be standardized for all points
    assert np.all(fun_bedpe["extrusion_x"] >= fun_bedpe["extrusion_y"])

    # the main table hosts the "start" and "end" positions, which is the main diagonal start and end
    # to get a single point for the jet, we can use the midpoint of the start and end positions
    fun_table["root"] = (fun_table["start"] + fun_table["end"]) / 2
    # Note that "root" position should technically be "root_x" and "root_y" but since 
    # this is a point on the main diagonal, they are the same and we can just use one value

    # Lets add a unique identifier now
    fun_table["unique_id"] = fun_table.index

    # Lets combine them
    fun_minimal = fun_bedpe.copy()

    fun_minimal["root"] = fun_table["root"]

    # Lets add a unique identifier now
    fun_minimal["unique_id"] = fun_minimal.index
    fun_minimal["chrom"] = fun_table["chrom"]

    fun_minimal = fun_minimal[["unique_id", "chrom", "root", "extrusion_x", "extrusion_y"]]

    # NEW
    _, result_dict = find_intersections(df_pred_in=fun_minimal, df_true_in=df_true, leniency=leniency)

    exp_name = os.path.basename(f_hic_file).strip(".hic")

    result_dict_pool["n_true"]     += len(df_true) if df_true is not None else 0
    result_dict_pool["n_reported"] += len(fun_minimal)

    for key in result_dict_pool.keys():
        if key in result_dict:
            result_dict_pool[key] += result_dict[key]


# Compute the aggregate precision, recall, and F1 score across all experiments
result_dict_pool["precision"], result_dict_pool["recall"], result_dict_pool["f1"] = precision_recall_f1(
    tp=result_dict_pool["tp"],
    fp=result_dict_pool["fp"],
    fn=result_dict_pool["fn"],
)
    
print(f"Pooled FUN performance across all experiments\n"
      f"* Resolution {resolution}, Leniency: ±{leniency}\n"
      f"* N true: {result_dict_pool['n_true']}, N reported: {result_dict_pool['n_reported']}\n"
      f"* TP: {result_dict_pool['tp']}, FP: {result_dict_pool['fp']}, FN: {result_dict_pool['fn']}\n"
      f"* F1: {result_dict_pool['f1']:.2f}, Precision: {result_dict_pool['precision']:.2f}, Recall: {result_dict_pool['recall']:.2f}")

Pooled FUN performance across all experiments
* Resolution 25000, Leniency: ±50000.0
* N true: 91, N reported: 70
* TP: 59, FP: 11, FN: 32
* F1: 0.73, Precision: 0.84, Recall: 0.65
